在深度学习的混合精度训练中，FP32 主权重副本（Master Weights）是保证模型能够稳定、精准训练的一个关键机制。

## 它是什么？
简单来说，<font color='red'>它是模型参数的一个高精度“账本”。</font>

在现代大模型训练中，为了提升计算速度并节省显存，我们通常会使用 FP16（半精度，占2字节）来进行模型的前向和反向计算。但是，FP16 的数值表示范围较小，精度较低。如果直接用 FP16 来更新参数，微小的梯度变化很容易因为精度不够而被直接舍入成 0（即“梯度下溢”），导致模型参数无法得到有效更新，训练失败。

为了解决这个问题，优化器（如 Adam）会额外维护一份 FP32（单精度，占4字节） 的参数副本。它的核心工作流程如下：
1. 计算时：使用轻量级的 FP16 参数进行前向和反向传播，算出 FP16 的梯度。
2. 更新时：将 FP16 梯度转换为 FP32，然后在高精度的 FP32 主权重副本上进行参数更新（加上动量、学习率缩放等操作）。
3. 同步时：将更新后的 FP32 主权重转换回 FP16，覆盖掉原来的 FP16 参数，用于下一轮的计算。

## 举个具体的例子：
假设你的模型中有一个参数（权重）当前的数值是 1.0000。
### 场景一：没有 FP32 主权重副本（纯 FP16 训练）
某次反向传播计算出的梯度非常微小，比如是 0.00005。由于 FP16 的精度有限，它可能无法准确表示这么小的数值，直接把它四舍五入成了 0。
- 更新公式：新权重 = 1.0000 - 学习率 × 0
- 结果：参数完全没变，模型学不到东西。

### 场景二：拥有 FP32 主权重副本（混合精度训练）
- 计算：<font color='red'>前向/反向传播依然用 FP16 算出梯度约为 0.00005。</font>
- 转换：在更新前，系统把这个微小的梯度转换为 FP32 格式，FP32 能够非常精准地记录 0.00005 这个数值。
- 更新：<font color='red'>优化器在它维护的 FP32 主权重副本（1.00000000）上进行计算。</font>
    - 假设学习率是 1，那么 1.00000000 - 0.00005 = 0.99995000。<font color='red'>FP32 完美地保存了这次微小的变化。</font>
- 同步：最后，把更新好的 FP32 主权重 0.99995000 转换回 FP16（变成 0.99995），赋值给模型的 FP16 参数。

通过这个例子可以看出，<font color='red'>FP32 主权重副本就像一个“高精度账本”，专门用来累积那些 FP16 无法承载的微小更新，确保模型在享受 FP16 带来的速度红利的同时，不会因为精度损失而走偏或停止学习。</font>

### 补充一点显存常识：
正因为 Adam 优化器需要为每个参数维护这份 FP32 主权重副本，再加上 FP32 格式的动量（一阶矩）和方差（二阶矩），导致 Adam 的优化器状态非常占用显存（每参数高达 12 字节的 FP32 状态），这也是为什么在微调大模型时，显存往往会被优化器状态占满的主要原因。